In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv("/content/ai4i2020.csv")
df.head()

In [ ]:
df.describe()

In [ ]:
df.tail()
df.isnull()

In [ ]:
df.info()

In [ ]:
df.shape


In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df['Machine failure'].value_counts

In [ ]:
sns.countplot(data=df, x="Machine failure")
plt.title("Machine failure")
plt.show()

In [ ]:
sns.heatmap(data=df.corr(numeric_only=True), annot=True)
plt.show()

In [ ]:
df.groupby("Machine failure").mean(numeric_only=True)

In [ ]:
sns.boxplot(data=df, x="Machine failure", y="Torque [Nm]")
plt.title("Torque vs Machine Failure")
plt.show()

In [ ]:
sns.boxplot(data=df, x="Machine failure", y="Rotational speed [rpm]")
plt.title("Rotational Speed vs Machine Failure")
plt.show()

In [ ]:
sns.boxplot(data=df, x="Machine failure", y="Tool wear [min]")
plt.title("Tool Wear vs Machine Failure")
plt.show()

In [ ]:
df.groupby("Machine failure")["Tool wear [min]"].agg(
    ["count", "mean", "median", "min", "max"]
)

In [ ]:
sns.boxplot(
    data=df,
    x="Machine failure",
    y="Tool wear [min]"
)

plt.title("Tool Wear vs Machine Failure")
plt.show()

In [ ]:
X = df[
    [
        "Type",
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]"
    ]
]

y = df["Machine failure"]

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Re-create X_train and X_test as DataFrames from the original X and y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply one-hot encoding to the 'Type' column
X_train = pd.get_dummies(X_train, columns=['Type'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['Type'], drop_first=True)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
model1=DecisionTreeClassifier()
model1.fit(X_train,y_train)
y_pred=model1.predict(X_test)
y_pred_train=model1.predict(X_train)
print(accuracy_score(y_train,y_pred_train))
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model2=RandomForestClassifier()
model2.fit(X_train,y_train)
y_pred=model2.predict(X_test)
y_pred_train=model2.predict(X_train)
print(accuracy_score(y_train,y_pred_train))
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

In [ ]:
from sklearn.svm import SVC
model3=SVC()
model3.fit(X_train,y_train)
y_pred=model3.predict(X_test)
y_pred_train=model3.predict(X_train)
print(accuracy_score(y_train,y_pred_train))
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

Here the Decision Tree has been the best model for this dataset.

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}


grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)
print("Best Hyperparameters:", grid_search.best_params_)
print("Best F1-Score from CV:", grid_search.best_score_)

In [ ]:
best_tree = grid_search.best_estimator_
y_pred_tuned = best_tree.predict(X_test)

print("Tuned Decision Tree Performance on Test Set:\n")
print(f"Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

In [ ]:
%%bash
pip install mlflow streamlit pyngrok -q

In [ ]:
import mlflow
import mlflow.sklearn
mlflow.set_experiment("Machine_Failure_Prediction")

with mlflow.start_run(run_name="Tuned_Decision_Tree"):
    # Log hyperparameters
    mlflow.log_params(grid_search.best_params_)

    # Calculate metrics
    test_accuracy = accuracy_score(y_test, y_pred_tuned)

    # Log metrics
    mlflow.log_metric("accuracy", test_accuracy)

    # Log model by adding the Tree structure to skops_trusted_types
    mlflow.sklearn.log_model(
        sk_model=best_tree,
        artifact_path="decision_tree_model",
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

    print("Model, parameters, and metrics have been logged to MLflow successfully!")

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import mlflow.sklearn
import joblib

# Set page config
st.set_page_config(page_title="Predictive Maintenance", layout="centered")

st.title("🛠️ Predictive Maintenance - Machine Failure App")
st.write("Enter the operational parameters below to predict whether a machine failure is likely to occur.")

# Define inputs
type_option = st.selectbox("Product Type (L: Low, M: Medium, H: High)", ["L", "M", "H"])
air_temp = st.number_input("Air Temperature [K]", min_value=200.0, max_value=400.0, value=298.2)
process_temp = st.number_input("Process Temperature [K]", min_value=200.0, max_value=400.0, value=308.6)
rotational_speed = st.number_input("Rotational Speed [rpm]", min_value=500, max_value=5000, value=1433)
torque = st.number_input("Torque [Nm]", min_value=0.0, max_value=150.0, value=39.5)
tool_wear = st.number_input("Tool Wear [min]", min_value=0, max_value=500, value=7)

if st.button("Predict Failure"):
    # Map inputs to match preparation pipeline
    input_data = {
        'Air temperature [K]': air_temp,
        'Process temperature [K]': process_temp,
        'Rotational speed [rpm]': rotational_speed,
        'Torque [Nm]': torque,
        'Tool wear [min]': tool_wear,
        'Type_L': 1 if type_option == 'L' else 0,
        'Type_M': 1 if type_option == 'M' else 0
    }

    input_df = pd.DataFrame([input_data])

    # Ensure we arrange the columns in the exact order expected by the model scaler
    # We can load the fitted scaler from our notebook environment or simulate scaling here
    # To keep it self-contained, we load the trained model.
    try:
        # Load the logged MLflow model
        # In a full MLops context, you'd specify the runs URI
        # Here we retrieve the globally trained 'best_tree' and 'scaler' from context
        pass
    except Exception as e:
        st.error(f"Error loading assets: {e}")

In [ ]:
import joblib
# Save the fitted scaler so the Streamlit app can use it to scale incoming features
joblib.dump(scaler, 'scaler.pkl')
print("Scaler successfully saved as scaler.pkl!")

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import mlflow.sklearn
import joblib

# Set page configuration
st.set_page_config(page_title="Predictive Maintenance", page_icon="🛠️", layout="centered")

st.title("🛠️ Predictive Maintenance - Machine Failure App")
st.write("Enter the operational parameters below to predict whether a machine failure is likely to occur.")

# Sidebar/Resource loading
@st.cache_resource
def load_assets():
    # Load the scaler
    scaler = joblib.load('scaler.pkl')
    # Retrieve our latest logged MLflow model run
    import mlflow
    # Retrieve runs for the current experiment
    experiment = mlflow.get_experiment_by_name("Machine_Failure_Prediction")
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    # Select the latest run
    latest_run_id = runs.iloc[0]['run_id']
    model_uri = f"runs:/{latest_run_id}/decision_tree_model"
    model = mlflow.sklearn.load_model(model_uri)
    return scaler, model

try:
    scaler, model = load_assets()
    st.success("Model and Scaler loaded successfully from MLflow & local files!")
except Exception as e:
    st.error(f"Error loading assets: {e}")
    st.stop()

# User Input form
with st.form("prediction_form"):
    col1, col2 = st.columns(2)
    with col1:
        type_option = st.selectbox("Product Type", ["L (Low)", "M (Medium)", "H (High)"])
        air_temp = st.number_input("Air Temperature [K]", min_value=200.0, max_value=400.0, value=298.2)
        process_temp = st.number_input("Process Temperature [K]", min_value=200.0, max_value=400.0, value=308.6)
    with col2:
        rotational_speed = st.number_input("Rotational Speed [rpm]", min_value=500, max_value=5000, value=1433)
        torque = st.number_input("Torque [Nm]", min_value=0.0, max_value=150.0, value=39.5)
        tool_wear = st.number_input("Tool Wear [min]", min_value=0, max_value=500, value=7)

    submit_button = st.form_submit_button("Predict Failure")

if submit_button:
    # Map inputs to match preparation pipeline columns:
    # ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_L', 'Type_M']
    type_val = type_option.split()[0]

    input_data = {
        'Air temperature [K]': air_temp,
        'Process temperature [K]': process_temp,
        'Rotational speed [rpm]': rotational_speed,
        'Torque [Nm]': torque,
        'Tool wear [min]': tool_wear,
        'Type_L': 1 if type_val == 'L' else 0,
        'Type_M': 1 if type_val == 'M' else 0
    }

    # Create DataFrame
    input_df = pd.DataFrame([input_data])

    # Standardize input variables using our saved scaler
    scaled_features = scaler.transform(input_df)

    # Perform prediction
    prediction = model.predict(scaled_features)
    probability = model.predict_proba(scaled_features)[0][1]

    # Output UI results
    st.subheader("Prediction Results:")
    if prediction[0] == 1:
        st.error(f"🚨 Machine Failure Predicted! (Failure Probability: {probability:.2%})")
    else:
        st.success(f"✅ Machine is Operating Normally (Failure Probability: {probability:.2%})")